In [1]:
import os
import re
import glob
import argparse
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from urllib.parse import urlencode
from datetime import datetime
from rasterstats import zonal_stats

In [2]:
def fetch_tifs(dataset, start_year=1991, end_year=datetime.now().year, month=8):
    """Download August GeoTIFFs for a dataset from start_year–end_year."""
    info = DATASETS[dataset]
    headers = {"Authorization": f"Bearer {API_KEY}"}
    os.makedirs(RASTER_DIR, exist_ok=True)

    for year in range(start_year, end_year + 1):
        params = info["params"].copy()
        params["date"] = f"{year}-{month:02d}-01"
        query = urlencode(params)
        url = f"{info['url']}?{query}"
        out_path = os.path.join(RASTER_DIR, f"{dataset}_{year}_{month:02d}.tif")

        if os.path.exists(out_path):
            print(f"Skipping {out_path} (already exists)")
            continue

        print(f"Fetching {dataset} for {params['date']}...")
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(res.content)
            print(f"Saved {out_path}")
        else:
            print(f"{res.status_code} for {url}")

In [3]:
def convert_units(value, dataset):
    """Convert rainfall mm→inches and temperature °C→°F."""
    if value is None or np.isnan(value):
        return np.nan
    if dataset == "rainfall":
        return value / 25.4  # mm → inches
    elif dataset == "temperature":
        return (value * 9/5) + 32  # °C → °F
    return value

In [4]:
DATASETS = {
    "rainfall": {
        "url": "https://api.hcdp.ikewai.org/raster",
        "climo_url": "./data/climo/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif",
        "params": {"datatype": "rainfall", "production": "new", "period": "month"},
    },
    "temperature": {
        "url": "https://api.hcdp.ikewai.org/raster",
        "climo_url": "./data/climo/temperature/monthly_air_temp_clim_statewide_1991-2020_8.tif",
        "params": {"datatype": "temperature", "aggregation": "mean", "period": "month"},
    },
}

DATE = datetime(2025, 8, 1)
MONTH = DATE.month
RASTER_DIR = "./data"

def get_key_from_environment(file_path, key):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)
    return match.group(1) if match else None


API_KEY = get_key_from_environment("../src/environments/environment.ts", "apiToken")

In [5]:
dataset = "temperature"
raster_folder = f"{RASTER_DIR}/"
climo_file = DATASETS[dataset]["climo_url"]
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi1_2025_08.tif"

print(f"\n--- Processing statewide ({dataset}) ---")

# --- Load climatology ---
with rasterio.open(climo_file) as src:
    clim = src.read(1).astype(float)
    clim = np.where(src.nodata == src.read(1), np.nan, clim)
    climo_mean = convert_units(np.nanmean(clim), dataset)

    # --- Loop through all historical rasters to get mean and anomaly ---
all_records = []
for tif in sorted(glob.glob(os.path.join(raster_folder, f"{dataset}_*_08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year, month = parts[1], parts[2]
    date = f"{year}-{month}"

    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        arr = np.where(arr == src.nodata, np.nan, arr)
        mean_val = convert_units(np.nanmean(arr), dataset)

    anomaly = mean_val - climo_mean
    pchange = ((mean_val - climo_mean) / climo_mean) * 100 if dataset == "rainfall" else anomaly

    all_records.append({
        "date": date,
        "year": int(year),
        "mean": mean_val,
        "anomaly": anomaly,
        "pchange": pchange,
    })

df = pd.DataFrame(all_records)

# --- Rank logic (dry for rainfall, warm for temperature) ---
ascending = True if dataset == "rainfall" else False
df["rank"] = df["mean"].rank(method="min", ascending=ascending)

df


--- Processing statewide (temperature) ---


,date,year,mean,anomaly,pchange,rank
0,1991-08,1991,69.084291,-0.384951,-0.384951,22.0
1,1992-08,1992,68.814223,-0.655019,-0.655019,25.0
2,1993-08,1993,68.714652,-0.754590,-0.754590,28.0
3,1994-08,1994,69.278251,-0.190991,-0.190991,21.0
4,1995-08,1995,68.313046,-1.156196,-1.156196,33.0
5,1996-08,1996,68.907691,-0.561551,-0.561551,24.0
6,1997-08,1997,68.763907,-0.705335,-0.705335,26.0
7,1998-08,1998,68.115152,-1.354090,-1.354090,35.0
8,1999-08,1999,68.571153,-0.898089,-0.898089,29.0
9,2000-08,2000,69.355905,-0.113338,-0.113338,18.0


In [13]:
dataset = "temperature"
fetch_tifs(dataset, start_year=1991, end_year=2025, month=8)

Fetching temperature for 1991-08-01...
Saved ./data/temperature_1991_08.tif
Fetching temperature for 1992-08-01...
Saved ./data/temperature_1992_08.tif
Fetching temperature for 1993-08-01...
Saved ./data/temperature_1993_08.tif
Fetching temperature for 1994-08-01...
Saved ./data/temperature_1994_08.tif
Fetching temperature for 1995-08-01...
Saved ./data/temperature_1995_08.tif
Fetching temperature for 1996-08-01...
Saved ./data/temperature_1996_08.tif
Fetching temperature for 1997-08-01...
Saved ./data/temperature_1997_08.tif
Fetching temperature for 1998-08-01...
Saved ./data/temperature_1998_08.tif
Fetching temperature for 1999-08-01...
Saved ./data/temperature_1999_08.tif
Fetching temperature for 2000-08-01...
Saved ./data/temperature_2000_08.tif
Fetching temperature for 2001-08-01...
Saved ./data/temperature_2001_08.tif
Fetching temperature for 2002-08-01...
Saved ./data/temperature_2002_08.tif
Fetching temperature for 2003-08-01...
Saved ./data/temperature_2003_08.tif
Fetching tem

In [6]:
get_statewide_stats(dataset)

NameError: name 'get_statewide_stats' is not defined

In [2]:
with rasterio.open("/Users/cherryleheu/Documents/HCDP/Data/climo/monthly/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif") as src:
    rf = src.read(1)
    nodata_climo = src.nodata

masked_climo = np.ma.masked_equal(rf, nodata_climo)
masked_climo_avg = masked_climo.mean()/25.4

In [14]:
with rasterio.open("/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/rainfall_2025_08.tif") as src:
    rf = src.read(1)
    nodata = src.nodata

In [3]:
#Statewide
raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climatology_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/monthly/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif"
records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, "rainfall*_08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year, month = parts[1], parts[2]
    date = f"{year}-{month}"
    with rasterio.open(tif) as src:
        rf = src.read(1)
        nodata = src.nodata

    masked_data = np.ma.masked_equal(rf, nodata)
    masked_data_avg = masked_data.mean()/25.4

    records.append({
        "division_full": "Statewide",
        "date": date,
        "mean": masked_data_avg,
        "anomaly": masked_data_avg - masked_climo_avg,
        "pchange": ((masked_data_avg - masked_climo_avg) / masked_climo_avg) * 100
    })

df = pd.DataFrame(records)

df["rank"] = df["anomaly"].rank(method="min")

this_month = df.iloc[[-1]]


tif_path = f'/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi1_2025_08.tif'

with rasterio.open(tif_path) as src:
    data = src.read(1)  
    mask = src.read_masks(1)  

nodata = src.nodata
if nodata is not None:
    data = np.where(data == nodata, np.nan, data)

valid_pixels = np.isfinite(data)
below_threshold = (data < 0.5)

percentage = (np.sum(below_threshold & valid_pixels) / np.sum(valid_pixels)) * 100

print(f"Percentage of pixels < 0 {percentage:.2f}%")
this_month['dry_pct'] = percentage
this_month.to_csv("../public/statewide_rainfall_stats.csv", index=False)
this_month

Percentage of pixels < 0 99.67%


/var/folders/vl/70ggslts0x98b_vgphfybj140000gn/T/ipykernel_9332/4154323640.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  this_month['dry_pct'] = percentage


,division_full,date,mean,anomaly,pchange,rank,dry_pct
35,Statewide,2025-08,1.971096,-3.189855,-61.807507,1.0,99.666647


In [123]:
#County
shapefile = "../public/shapefiles/county.shp"
raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climatology_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/monthly/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi1_2025_08.tif"

gdf = gpd.read_file(shapefile)

possible_island_cols = ["island", "ISLAND", "mokupuni", "Mokupuni"]
island_col = next(
    (c for c in gdf.columns if c.lower() in ["island", "mokupuni", "isle", "islandname"]),
    None
)
name_col = next(
    (c for c in gdf.columns if c.lower() in ["name", "division", "moku", "climate_div","ahupuaa", "county"]),
    None
)

if island_col and name_col:
    print(f"Using island_col='{island_col}', name_col='{name_col}'")  
    gdf = gdf.dissolve(by=[island_col, name_col], as_index=False)
    gdf["division_full"] = gdf.apply(
        lambda r: f"{r[island_col]}::{r[name_col]}", axis=1
    )
elif name_col:
    print(f"No island column found — using only '{name_col}'")
    gdf = gdf.dissolve(by=name_col, as_index=False)
    gdf["division_full"] = gdf[name_col]
else:
    raise ValueError(
        "No valid island or name column found in shapefile. "
        "Check attribute table for field names."
    )

nodata_climo = None
climo_stats = zonal_stats(vectors=gdf, raster=climatology_file, stats=["mean"], nodata=nodata_climo)
gdf["climo_mean"] = [
    (c["mean"] / 25.4) if c["mean"] is not None else np.nan
    for c in climo_stats
]


all_records = []
nodata = None

for tif in sorted(glob.glob(os.path.join(raster_folder, "rainfall*_08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year, month = parts[1], parts[2]
    date = f"{year}-{month}"

    if year == "1990":
        continue

    stats = zonal_stats(vectors=gdf, raster=tif, stats=["mean"], nodata=nodata)

    for idx, division in gdf.iterrows():
        mean_raw = stats[idx]["mean"]
        if mean_raw is None or np.isnan(mean_raw):
            mean_val = np.nan
            anomaly = np.nan
            pchange = np.nan
        else:
            mean_val = mean_raw / 25.4
            climo_mean = division["climo_mean"]
            if np.isnan(climo_mean):
                anomaly = np.nan
                pchange = np.nan
            else:
                anomaly = mean_val - climo_mean
                pchange = ((mean_val - climo_mean) / climo_mean) * 100

        all_records.append({
            "division_full": division["division_full"],
            "date": date,
            "mean": mean_val,
            "anomaly": anomaly,
            "pchange": pchange
        })

with rasterio.open(tif_path) as src:
    data = src.read(1)
    mask = src.read_masks(1)
    nodata = src.nodata
    data = np.where((mask == 0) | (data == nodata), np.nan, data)

def pct_less_than_half(values):
    vals = np.array(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    return (np.sum(vals < -0.5) / len(vals)) * 100

percent_stats = zonal_stats(
    vectors=gdf,
    raster=tif_path,
    stats=None,
    add_stats={"pct_drought": pct_less_than_half},
    nodata=nodata
)

gdf["pct_drought"] = [s["pct_drought"] for s in percent_stats]

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("division_full")["anomaly"].rank(method="min")

merged = df.merge(
    gdf[["division_full", "pct_drought"]],
    on="division_full",
    how="left"
)

aug2025_df = merged[merged["date"] == "2025-08"].reset_index(drop=True)

out_csv = "../public/county_rainfall_stats.csv"
aug2025_df.to_csv(out_csv, index=False)

aug2025_df

Using island_col='isle', name_col='county'


,division_full,date,mean,anomaly,pchange,rank,pct_drought
0,Hawaiʻi::Hawaiʻi,2025-08,2.440614,-3.357301,-57.905316,4.0,60.957475
1,Kahoʻolawe::Maui,2025-08,0.175651,-0.329943,-65.258514,13.0,73.443798
2,Kauaʻi::Kauaʻi,2025-08,1.969639,-3.636947,-64.869193,2.0,94.119255
3,Lānaʻi::Maui,2025-08,0.213612,-0.473973,-68.932953,6.0,100.000000
4,Maui::Maui,2025-08,1.351539,-3.745549,-73.484099,1.0,73.799178
5,Molokaʻi::Maui,2025-08,0.865151,-1.703074,-66.313273,4.0,76.498256
6,Oʻahu::Honolulu,2025-08,0.887214,-2.898517,-76.564262,1.0,100.000000


In [8]:
#Moku
shapefile = "../public/shapefiles/moku.shp"
raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climatology_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/monthly/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi1_2025_08.tif"

gdf = gpd.read_file(shapefile)

possible_island_cols = ["island", "ISLAND", "mokupuni", "Mokupuni"]
island_col = next(
    (c for c in gdf.columns if c.lower() in ["island", "mokupuni", "isle", "islandname"]),
    None
)
name_col = next(
    (c for c in gdf.columns if c.lower() in ["name", "division", "moku", "climate_div","ahupuaa"]),
    None
)

if island_col and name_col:
    print(f"Using island_col='{island_col}', name_col='{name_col}'")  
    gdf = gdf.dissolve(by=[island_col, name_col], as_index=False)
    gdf["division_full"] = gdf.apply(
        lambda r: f"{r[island_col]}::{r[name_col]}", axis=1
    )
elif name_col:
    print(f"No island column found — using only '{name_col}'")
    gdf = gdf.dissolve(by=name_col, as_index=False)
    gdf["division_full"] = gdf[name_col]
else:
    raise ValueError(
        "No valid island or name column found in shapefile. "
        "Check attribute table for field names."
    )

nodata_climo = None
climo_stats = zonal_stats(vectors=gdf, raster=climatology_file, stats=["mean"], nodata=nodata_climo)
gdf["climo_mean"] = [
    (c["mean"] / 25.4) if c["mean"] is not None else np.nan
    for c in climo_stats
]


all_records = []
nodata = None

for tif in sorted(glob.glob(os.path.join(raster_folder, "rainfall*_08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year, month = parts[1], parts[2]
    date = f"{year}-{month}"

    if year == "1990":
        continue

    stats = zonal_stats(vectors=gdf, raster=tif, stats=["mean"], nodata=nodata)

    for idx, division in gdf.iterrows():
        mean_raw = stats[idx]["mean"]
        if mean_raw is None or np.isnan(mean_raw):
            mean_val = np.nan
            anomaly = np.nan
            pchange = np.nan
        else:
            mean_val = mean_raw / 25.4
            climo_mean = division["climo_mean"]
            if np.isnan(climo_mean):
                anomaly = np.nan
                pchange = np.nan
            else:
                anomaly = mean_val - climo_mean
                pchange = ((mean_val - climo_mean) / climo_mean) * 100

        all_records.append({
            "division_full": division["division_full"],
            "date": date,
            "mean": mean_val,
            "anomaly": anomaly,
            "pchange": pchange
        })

with rasterio.open(tif_path) as src:
    data = src.read(1)
    mask = src.read_masks(1)
    nodata = src.nodata
    data = np.where((mask == 0) | (data == nodata), np.nan, data)

def pct_less_than_half(values):
    vals = np.array(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    return (np.sum(vals < -0.5) / len(vals)) * 100

percent_stats = zonal_stats(
    vectors=gdf,
    raster=tif_path,
    stats=None,
    add_stats={"pct_drought": pct_less_than_half},
    nodata=nodata
)

gdf["pct_drought"] = [s["pct_drought"] for s in percent_stats]

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("division_full")["anomaly"].rank(method="min")

merged = df.merge(
    gdf[["division_full", "pct_drought"]],
    on="division_full",
    how="left"
)

aug2025_df = merged[merged["date"] == "2025-08"].reset_index(drop=True)

out_csv = "../public/rainfall/moku_rainfall_stats.csv"
aug2025_df.to_csv(out_csv, index=False)

aug2025_df

Using island_col='mokupuni', name_col='moku'


,division_full,date,mean,anomaly,pchange,rank,pct_drought
0,Hawaiʻi::Hilo,2025-08,4.448600,-8.239161,-64.937863,3.0,91.178244
1,Hawaiʻi::Hāmākua,2025-08,1.428596,-2.550709,-64.099355,4.0,70.629855
2,Hawaiʻi::Kaʻū,2025-08,2.061234,-1.797155,-46.577855,15.0,44.410784
3,Hawaiʻi::Kohala,2025-08,1.222980,-1.552920,-55.942926,3.0,52.120851
4,Hawaiʻi::Kona,2025-08,1.499498,-0.854633,-36.303532,9.0,42.517952
5,Hawaiʻi::Puna,2025-08,4.313927,-6.503159,-60.119325,3.0,93.849291
6,Kahoʻolawe::Honuaʻula,2025-08,0.344777,-0.506618,-59.504485,13.0,52.736087
7,Kauaʻi::Halele'a,2025-08,3.298999,-7.229656,-68.666471,2.0,100.000000
8,Kauaʻi::Kona,2025-08,0.912653,-2.603539,-74.044275,1.0,98.269768
9,Kauaʻi::Koʻolau,2025-08,1.977895,-3.101606,-61.061234,5.0,100.000000


In [9]:
#Ahupuaa
shapefile = "../public/shapefiles/ahupuaa.shp"
raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climatology_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/monthly/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi1_2025_08.tif"

gdf = gpd.read_file(shapefile)

possible_island_cols = ["island", "ISLAND", "mokupuni", "Mokupuni"]
island_col = next(
    (c for c in gdf.columns if c.lower() in ["island", "mokupuni", "isle", "islandname"]),
    None
)
name_col = next(
    (c for c in gdf.columns if c.lower() in ["name", "division", "moku", "climate_div","ahupuaa"]),
    None
)

if island_col and name_col:
    print(f"Using island_col='{island_col}', name_col='{name_col}'")  
    gdf = gdf.dissolve(by=[island_col, name_col], as_index=False)
    gdf["division_full"] = gdf.apply(
        lambda r: f"{r[island_col]}::{r[name_col]}", axis=1
    )
elif name_col:
    print(f"No island column found — using only '{name_col}'")
    gdf = gdf.dissolve(by=name_col, as_index=False)
    gdf["division_full"] = gdf[name_col]
else:
    raise ValueError(
        "No valid island or name column found in shapefile. "
        "Check attribute table for field names."
    )

nodata_climo = None
climo_stats = zonal_stats(vectors=gdf, raster=climatology_file, stats=["mean"], nodata=nodata_climo)
gdf["climo_mean"] = [
    (c["mean"] / 25.4) if c["mean"] is not None else np.nan
    for c in climo_stats
]


all_records = []
nodata = None

for tif in sorted(glob.glob(os.path.join(raster_folder, "rainfall*_08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year, month = parts[1], parts[2]
    date = f"{year}-{month}"

    if year == "1990":
        continue

    stats = zonal_stats(vectors=gdf, raster=tif, stats=["mean"], nodata=nodata)

    for idx, division in gdf.iterrows():
        mean_raw = stats[idx]["mean"]
        if mean_raw is None or np.isnan(mean_raw):
            mean_val = np.nan
            anomaly = np.nan
            pchange = np.nan
        else:
            mean_val = mean_raw / 25.4
            climo_mean = division["climo_mean"]
            if np.isnan(climo_mean):
                anomaly = np.nan
                pchange = np.nan
            else:
                anomaly = mean_val - climo_mean
                pchange = ((mean_val - climo_mean) / climo_mean) * 100

        all_records.append({
            "division_full": division["division_full"],
            "date": date,
            "mean": mean_val,
            "anomaly": anomaly,
            "pchange": pchange
        })

with rasterio.open(tif_path) as src:
    data = src.read(1)
    mask = src.read_masks(1)
    nodata = src.nodata
    data = np.where((mask == 0) | (data == nodata), np.nan, data)

def pct_less_than_half(values):
    vals = np.array(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    return (np.sum(vals < -0.5) / len(vals)) * 100

percent_stats = zonal_stats(
    vectors=gdf,
    raster=tif_path,
    stats=None,
    add_stats={"pct_drought": pct_less_than_half},
    nodata=nodata
)

gdf["pct_drought"] = [s["pct_drought"] for s in percent_stats]

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("division_full")["anomaly"].rank(method="min")

merged = df.merge(
    gdf[["division_full", "pct_drought"]],
    on="division_full",
    how="left"
)

aug2025_df = merged[merged["date"] == "2025-08"].reset_index(drop=True)

out_csv = "../public/rainfall/ahupuaa_rainfall_stats.csv"
aug2025_df.to_csv(out_csv, index=False)

aug2025_df

Using island_col='mokupuni', name_col='ahupuaa'


,division_full,date,mean,anomaly,pchange,rank,pct_drought
0,"Hawaiʻi::Ahalanui, Laepaoʻo",2025-08,3.158976,-4.595310,-59.261549,2.0,100.000000
1,"Hawaiʻi::Alaeakila, Kaapahu",2025-08,2.449805,-5.295233,-68.369361,3.0,100.000000
2,Hawaiʻi::Alakahi,2025-08,5.606875,-12.212520,-68.534987,1.0,100.000000
3,Hawaiʻi::Aleamai,2025-08,4.815272,-10.124845,-67.769517,1.0,100.000000
4,"Hawaiʻi::Anapuka, Hoʻopūloa",2025-08,1.521696,-0.494908,-24.541650,13.0,11.890244
...,...,...,...,...,...,...,...
691,Oʻahu::ʻŌhikilolo,2025-08,0.280069,-0.790022,-73.827541,6.0,100.000000
692,Oʻahu::ʻŌiʻo 1,2025-08,0.667778,-3.016066,-81.872798,1.0,100.000000
693,Oʻahu::ʻŌiʻo 2,2025-08,0.574975,-2.618224,-81.993759,1.0,100.000000
694,Oʻahu::ʻŌpana 1,2025-08,0.469596,-2.190808,-82.348693,1.0,100.000000


In [10]:
#Watershed
shapefile = "../public/shapefiles/watershed.shp"
raster_folder = "/Users/cherryleheu/Documents/HCDP/Data/monthly/rainfall/"
climatology_file = "/Users/cherryleheu/Documents/HCDP/Data/climo/monthly/rainfall/monthly_rainfall_clim_statewide_1991-2020_8.tif"
tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi1_2025_08.tif"

gdf = gpd.read_file(shapefile)

possible_island_cols = ["island", "ISLAND", "mokupuni", "Mokupuni"]
island_col = next(
    (c for c in gdf.columns if c.lower() in ["island", "mokupuni", "isle", "islandname"]),
    None
)
name_col = next(
    (c for c in gdf.columns if c.lower() in ["name", "division", "moku", "climate_div","ahupuaa"]),
    None
)

if island_col and name_col:
    print(f"Using island_col='{island_col}', name_col='{name_col}'")  
    gdf = gdf.dissolve(by=[island_col, name_col], as_index=False)
    gdf["division_full"] = gdf.apply(
        lambda r: f"{r[island_col]}::{r[name_col]}", axis=1
    )
elif name_col:
    print(f"No island column found — using only '{name_col}'")
    gdf = gdf.dissolve(by=name_col, as_index=False)
    gdf["division_full"] = gdf[name_col]
else:
    raise ValueError(
        "No valid island or name column found in shapefile. "
        "Check attribute table for field names."
    )

nodata_climo = None
climo_stats = zonal_stats(vectors=gdf, raster=climatology_file, stats=["mean"], nodata=nodata_climo)
gdf["climo_mean"] = [
    (c["mean"] / 25.4) if c["mean"] is not None else np.nan
    for c in climo_stats
]


all_records = []
nodata = None

for tif in sorted(glob.glob(os.path.join(raster_folder, "rainfall*_08.tif"))):
    parts = os.path.basename(tif).replace(".tif", "").split("_")
    year, month = parts[1], parts[2]
    date = f"{year}-{month}"

    if year == "1990":
        continue

    stats = zonal_stats(vectors=gdf, raster=tif, stats=["mean"], nodata=nodata)

    for idx, division in gdf.iterrows():
        mean_raw = stats[idx]["mean"]
        if mean_raw is None or np.isnan(mean_raw):
            mean_val = np.nan
            anomaly = np.nan
            pchange = np.nan
        else:
            mean_val = mean_raw / 25.4
            climo_mean = division["climo_mean"]
            if np.isnan(climo_mean):
                anomaly = np.nan
                pchange = np.nan
            else:
                anomaly = mean_val - climo_mean
                pchange = ((mean_val - climo_mean) / climo_mean) * 100

        all_records.append({
            "division_full": division["division_full"],
            "date": date,
            "mean": mean_val,
            "anomaly": anomaly,
            "pchange": pchange
        })

with rasterio.open(tif_path) as src:
    data = src.read(1)
    mask = src.read_masks(1)
    nodata = src.nodata
    data = np.where((mask == 0) | (data == nodata), np.nan, data)

def pct_less_than_half(values):
    vals = np.array(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    return (np.sum(vals < -0.5) / len(vals)) * 100

percent_stats = zonal_stats(
    vectors=gdf,
    raster=tif_path,
    stats=None,
    add_stats={"pct_drought": pct_less_than_half},
    nodata=nodata
)

gdf["pct_drought"] = [s["pct_drought"] for s in percent_stats]

df = pd.DataFrame(all_records)
df["rank"] = df.groupby("division_full")["anomaly"].rank(method="min")

merged = df.merge(
    gdf[["division_full", "pct_drought"]],
    on="division_full",
    how="left"
)

aug2025_df = merged[merged["date"] == "2025-08"].reset_index(drop=True)

out_csv = "../public/rainfall/watershed_rainfall_stats.csv"
aug2025_df.to_csv(out_csv, index=False)

aug2025_df

Using island_col='island', name_col='name'


,division_full,date,mean,anomaly,pchange,rank,pct_drought
0,Hawaiʻi::Aamakao Gulch,2025-08,2.262165,-3.126749,-58.021870,2.0,100.000000
1,Hawaiʻi::Aamanu,2025-08,2.965879,-5.132019,-63.374705,4.0,100.000000
2,Hawaiʻi::Ahole,2025-08,5.976032,-10.474269,-63.672204,4.0,100.000000
3,Hawaiʻi::Ahulua,2025-08,0.878895,-0.941903,-51.730225,9.0,38.169643
4,Hawaiʻi::Alaialoa Gulch,2025-08,3.173534,-8.122357,-71.905413,2.0,100.000000
...,...,...,...,...,...,...,...
983,Oʻahu::Wailupe,2025-08,0.675700,-2.912319,-81.167893,1.0,100.000000
984,Oʻahu::Waimalu,2025-08,1.648968,-4.770227,-74.311917,1.0,100.000000
985,Oʻahu::Waimanalo,2025-08,0.518820,-2.006702,-79.456921,1.0,100.000000
986,Oʻahu::Waimanalo Gulch,2025-08,0.279227,-1.186482,-80.949349,2.0,100.000000


In [122]:
import geopandas as gpd
gdf = gpd.read_file("../public/shapefiles/county.shp")
print(gdf.columns)


Index(['isle', 'county', 'lat', 'lon', 'geometry'], dtype='object')


In [131]:
import rasterio
import numpy as np

for m in range(1, 9):
    tif_path = f'/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi3_2025_0{m}.tif'

    with rasterio.open(tif_path) as src:
        data = src.read(1)  
        mask = src.read_masks(1)  

    nodata = src.nodata
    if nodata is not None:
        data = np.where(data == nodata, np.nan, data)

    valid_pixels = np.isfinite(data)
    below_threshold = (data < -0.5)


    percentage = (np.sum(below_threshold & valid_pixels) / np.sum(valid_pixels)) * 100

    print(f"Percentage of pixels < -1 in month {m}: {percentage:.2f}%")


Percentage of pixels < -1 in month 1: 34.84%
Percentage of pixels < -1 in month 2: 56.20%
Percentage of pixels < -1 in month 3: 33.93%
Percentage of pixels < -1 in month 4: 54.15%
Percentage of pixels < -1 in month 5: 38.35%
Percentage of pixels < -1 in month 6: 31.08%
Percentage of pixels < -1 in month 7: 41.80%
Percentage of pixels < -1 in month 8: 66.97%


In [136]:
import rasterio
import numpy as np

# Path to your GeoTIFF
for m in range(1, 9):
    tif_path = f'/Users/cherryleheu/Documents/HCDP/Data/monthly/SPI_historical/SPI_historical_new/spi3_2025_0{m}.tif'

    # Open and read the raster
    with rasterio.open(tif_path) as src:
        data = src.read(1)  # Read the first band
        mask = src.read_masks(1)  # Optional: read the mask to ignore nodata

    # Mask out nodata values
    nodata = src.nodata
    if nodata is not None:
        data = np.where(data == nodata, np.nan, data)

    # Compute percentage of pixels < -1 (excluding NaN)
    valid_pixels = np.isfinite(data)
    below_threshold = (data < -2)

    percentage = (np.sum(below_threshold & valid_pixels) / np.sum(valid_pixels)) * 100

    print(f"Percentage of pixels < -1 in month {m}: {percentage:.2f}%")


Percentage of pixels < -1 in month 1: 0.00%
Percentage of pixels < -1 in month 2: 7.29%
Percentage of pixels < -1 in month 3: 0.00%
Percentage of pixels < -1 in month 4: 4.88%
Percentage of pixels < -1 in month 5: 0.52%
Percentage of pixels < -1 in month 6: 2.97%
Percentage of pixels < -1 in month 7: 0.29%
Percentage of pixels < -1 in month 8: 2.27%
